## Settings

In [29]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [30]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid,
    compile_cross_valid,
    results_cross_valid
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initalization

In [31]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
data = load_processed_data()
models = load_estimators()

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

Original Data: 32 features, 25 observations


## Training

In [32]:
## leave-one-group-out cross validation (domain)
if "results_dict_domain" not in globals():
    results_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",  ## fold on domain
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_domain[name] = frontier


In [33]:
## leave-one-group-out cross validation (discipline)
if "results_dict_disc" not in globals():
    results_dict_disc = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "discipline",  ## fold on discipline
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_disc[name] = frontier


In [34]:
## 5-fold cross validation
if "results_dict_5fold" not in globals():
    results_dict_5fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 5,  ## fold on random 80-20 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_5fold[name] = frontier


In [35]:
## 10-fold cross validation
if "results_dict_10fold" not in globals():
    results_dict_10fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 10,  ## fold on random 90-10 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_10fold[name] = frontier


## Post-Processing

In [36]:
## convert all results to dataframes
results_data_domain = compile_cross_valid(results = results_dict_domain)
results_data_disc = compile_cross_valid(results = results_dict_disc)
results_data_5fold = compile_cross_valid(results = results_dict_5fold)
results_data_10fold = compile_cross_valid(results = results_dict_10fold)

## Cross-Validation Test
Four resampling regimes compare strict held-out-group transfer against random-split baselines. Domain and discipline LOGO test whether models trained on all remaining groups generalize to an unseen domain or discipline. Random 5-fold and 10-fold CV provide standard reference points.

- **Domain (LOGO)**: Leave-one-group-out by domain (5 groups). For each model, frontier metrics are averaged over held-out domains. Repeats vary learner initialization. The held-out domain splits stay fixed.
- **Discipline (LOGO)**: Leave-one-group-out by discipline (25 groups). For each model, frontier metrics are averaged over held-out disciplines. Repeats vary learner initialization. The held-out discipline splits stay fixed.
- **Random (5-Fold)**: Repeated shuffled 80–20 splits. For each model, frontier metrics are averaged over repeated random folds.
- **Random (10-Fold)**: Repeated shuffled 90–10 splits. For each model, frontier metrics are averaged over repeated random folds.

### Reporting Convention
Each regime contributes one model-level mean after averaging across groups or folds and repeats. The table reports the median of those model-level means across learners. EI is shown as median [Q1–Q3]. VR is violation rate. MV and MS remain on the log-target scale. All summaries are equally weighted across groups, folds, repeats, and models.

In [37]:
## compile transfer bounds and baseline benchmarks
results = results_cross_valid(
    {"Domain (LOGO)": results_data_domain, "Discipline (LOGO)": results_data_disc},
    {"Random (5-Fold)": results_data_5fold, "Random (10-Fold)": results_data_10fold},
    keys = ("Transfer", "Standard"),
    indicies = ("Evaluation", "Method"),
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = 3
)

display(results)

TypeError: results_cross_valid() got an unexpected keyword argument 'n_repeats'

Transfer EI remains substantial under held-out domains and disciplines. Transfer IQR spreads are narrow relative to their median EI. Cross-domain generalization is stable across learning paradigms. Baseline regimes confirm the expected upper bound under random partitioning. EI generalizes beyond the domains on which it was trained.